In [ ]:
# %% SUPER ENRICHMENT & WRITE-BACK PIPELINE
# (BB-Ticker handling, per-type country fill, parallel ISIN lookups, ISIN.SUFFIX filter, accurate metrics)

import pandas as pd
import requests
from pathlib import Path
import time
import re
import random
import yfinance as yf  # pip install yfinance
from concurrent.futures import ThreadPoolExecutor, as_completed

PARALLEL_WORKERS = 8  # threads for ISIN lookups

# ========= CONFIG =========
BASE_PATH = Path(r"D:\LinhDao\Programming\SUPERFUNdProject\final_data\vision_final.csv")
ENRICHED_PATH = BASE_PATH.with_name(BASE_PATH.stem + "_enriched.csv")

# Lookup tables
BBG_PATH = Path(r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\bloomberg-exchange-codes-full.csv")
YF_PATH  = Path(r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\yahoo_suffix_mapping_full.csv")

# OpenFIGI
OPENFIGI_API_KEY = "445ba62b-3be2-4f04-8f4f-4c9ec264d72b"
OPENFIGI_URL = "https://api.openfigi.com/v3/mapping"

# Run toggles
RUN_CLASSIFY = True
RUN_ISIN_TO_YAHOO = True
RUN_SEDOL_TO_EXCH_TICKER = True
RUN_COUNTRY_FROM_EXCHANGE_AND_YF = True
SAVE_ENRICHED = True
WRITE_COUNTRY_BACK_TO_BASE = True
OVERWRITE_EXISTING_COUNTRY = True


# ========= HELPERS =========

def _is_bb_ticker(val: str) -> bool:
    """
    BB Ticker pattern: '<symbol> <EX>' where EX is exactly two alnum chars (A-Z or 0-9).
    Examples: '000100 KS', 'BEI GR', 'ADNOCDIS UH', 'FOO C1'
    """
    s = str(val).strip()
    parts = s.split()
    if len(parts) != 2:
        return False
    ex = parts[-1].upper()
    return len(ex) == 2 and re.fullmatch(r"[A-Z0-9]{2}", ex) is not None

def classify_stockid(x: str) -> str:
    """
    ISIN: 12 chars, starts with 2 uppercase letters, no spaces
    SEDOL: 7 chars (alnum, no spaces) OR 5–6 digits (no spaces)
    BB Ticker: '<symbol> <EX>' with EX being two alnum chars (A–Z or 0–9)
    Else: empty/unknown
    """
    if x is None:
        return "EMPTY"
    val = str(x).strip()
    if val == "":
        return "EMPTY"

    no_space = " " not in val

    # ISIN
    if no_space and len(val) == 12 and val[:2].isalpha() and val[:2].isupper():
        return "ISIN"

    # Too-long no-space → unknown
    if no_space and len(val) > 12:
        return "EMPTY"

    # SEDOL
    if no_space:
        if len(val) == 7:
            return "SEDOL"
        if val.isdigit() and len(val) in (5, 6):  # 5–6 digit cores
            return "SEDOL"

    # BB Ticker
    if _is_bb_ticker(val):
        return "BB Ticker"

    return "EMPTY"

def yahoo_symbol_from_isin(isin: str) -> str | None:
    """
    Lighter resolve with retries:
      1) touch fast_info (cheap)
      2) read t.ticker — if it differs from the ISIN, treat as resolved
      3) final fallback on last attempt: get_info()['symbol'] (heavier)
    Jittered backoff to ride out transient 401s.
    """
    isin_u = str(isin).strip().upper()
    delays = [0.2, 0.4, 0.8, 1.6]  # seconds
    for attempt, delay in enumerate([0.0] + delays):
        if delay:
            time.sleep(delay + random.uniform(0, 0.15))
        try:
            t = yf.Ticker(isin_u)
            # nudge
            try:
                _ = t.fast_info
            except Exception:
                pass
            # light check
            sym = (t.ticker or "").strip()
            if sym and sym.upper() != isin_u:
                return sym
            # heavy fallback on last attempt
            if attempt == len(delays):
                try:
                    info = t.get_info()
                    if isinstance(info, dict):
                        s2 = str(info.get("symbol") or "").strip()
                        if s2:
                            return s2
                except Exception:
                    pass
        except Exception:
            continue
    return None

def yahoo_symbols_from_isins_parallel(isins: list[str], workers: int = PARALLEL_WORKERS) -> dict[str, str | None]:
    """Fetch Yahoo symbols for ISINs concurrently using yfinance.Ticker(ISIN)."""
    seen, uniq = set(), []
    for x in isins:
        if x not in seen:
            seen.add(x)
            uniq.append(x)

    out: dict[str, str | None] = {}
    if not uniq:
        return out

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(yahoo_symbol_from_isin, isin): isin for isin in uniq}
        for fut in as_completed(futs):
            isin = futs[fut]
            try:
                out[isin] = fut.result()
            except Exception:
                out[isin] = None
    return out

def clean_yahoo_symbol_from_isin(isin: str, sym: str | None) -> str:
    """
    Reject symbols that look like 'ISIN.<suffix>' (e.g., 'US36118L1061.SG').
    Keep normal tickers like 'FUTU' (or 'FUTU.*').
    """
    if not sym:
        return ""
    isin_u = str(isin).strip().upper()
    s = str(sym).strip()
    if s.upper().startswith(isin_u + "."):
        return ""  # drop Yahoo's ISIN-stub listings
    return s

def normalize_sedol_for_lookup(original: str) -> str:
    """
    Runtime-only normalization for OpenFIGI:
    - If 5 or 6 digits -> pad with leading zeros to 7.
    - Otherwise return as-is (7-char alnum SEDOL supported).
    DOES NOT change your CSV values.
    """
    s = str(original).strip().upper()
    if s.isdigit() and len(s) in (5, 6):
        return s.zfill(7)
    return s

def openfigi_map_sedols(original_sedols: list[str]) -> dict[str, dict]:
    """
    Returns: {original_sedol: {'ticker': <str>, 'exchCode': <str>}}
    Uses ID_SEDOL; 5–6 digit cores padded to 7 for lookup ONLY.
    """
    if not original_sedols:
        return {}
    headers = {"Content-Type": "application/json"}
    if OPENFIGI_API_KEY:
        headers["X-OPENFIGI-APIKEY"] = OPENFIGI_API_KEY

    norm_map = {orig: normalize_sedol_for_lookup(orig) for orig in original_sedols}
    rev_map = {}
    for orig, norm in norm_map.items():
        rev_map.setdefault(norm, []).append(orig)

    jobs = [{"idType": "ID_SEDOL", "idValue": norm} for norm in rev_map.keys()]
    out = {orig: {"ticker": "", "exchCode": ""} for orig in original_sedols}

    for i in range(0, len(jobs), 100):
        batch = jobs[i:i+100]
        try:
            r = requests.post(OPENFIGI_URL, json=batch, headers=headers, timeout=30)
            r.raise_for_status()
            data = r.json()
            for job, resp in zip(batch, data):
                norm = job["idValue"]
                originals = rev_map.get(norm, [])
                ticker = exch = ""
                if isinstance(resp, dict) and isinstance(resp.get("data"), list) and len(resp["data"]) > 0:
                    row = next((d for d in resp["data"] if d.get("ticker") and d.get("exchCode")), resp["data"][0])
                    ticker = str(row.get("ticker") or "").strip()
                    exch   = str(row.get("exchCode") or "").strip()
                for orig in originals:
                    out[orig] = {"ticker": ticker, "exchCode": exch}
        except requests.RequestException:
            pass
        time.sleep(0.15)
    return out

def first2(s: str) -> str:
    """First two characters (letters or digits), uppercased."""
    return str(s).strip().upper()[:2]

def extract_suffix(t: str) -> str:
    """Return Yahoo suffix INCLUDING the dot ('.AX'), or '' if none."""
    t = str(t).strip()
    return t[t.rfind("."):] if "." in t else ""

def build_bbg_map(bbg_df: pd.DataFrame) -> dict[str, str]:
    """(first2 of BBG_Code / Composite_Code) -> Country (Friendly)"""
    for c in ("BBG_Code", "Composite_Code", "Country (Friendly)"):
        if c not in bbg_df.columns:
            raise ValueError(f"Bloomberg file missing '{c}'. Headers: {list(bbg_df.columns)}")
    mapping = {}
    for _, r in bbg_df.iterrows():
        country = str(r["Country (Friendly)"]).strip()
        c1 = first2(r["BBG_Code"])
        if c1 and country and c1 not in mapping:
            mapping[c1] = country
        c2 = first2(r["Composite_Code"])
        if c2 and country and c2 not in mapping:
            mapping[c2] = country
    return mapping

def build_yahoo_map(yf_df: pd.DataFrame) -> dict[str, str]:
    """Suffix ('' allowed) -> Country."""
    for c in ("Suffix", "Country"):
        if c not in yf_df.columns:
            raise ValueError(f"Yahoo suffix file missing '{c}'. Headers: {list(yf_df.columns)}")
    return {str(r["Suffix"]).strip(): r["Country"] for _, r in yf_df.iterrows()}


# ========= PIPELINE =========

def run_enrichment_pipeline(
    base_path: Path,
    enriched_path: Path,
    bbg_path: Path,
    yf_path: Path,
    run_classify=True,
    run_isin=True,
    run_sedol=True,
    run_country=True,
    save_enriched=True,
):
    """
    Loads the base CSV, runs enrichment steps (toggleable), writes an *_enriched.csv for review,
    and returns (enriched_subset) containing only rows with non-empty Stock ID.
    """
    # Read as string to preserve leading zeros
    df = pd.read_csv(base_path, keep_default_na=False, dtype={"Stock ID": str})
    if "Stock ID" not in df.columns:
        raise ValueError("Base file missing 'Stock ID' column.")

    mask_has_value = df["Stock ID"].astype(str).str.strip().ne("")
    work = df.loc[mask_has_value].copy()

    # 1) Classify
    if run_classify:
        work["ID_Type"] = work["Stock ID"].apply(classify_stockid)
    else:
        if "ID_Type" not in work.columns:
            work["ID_Type"] = ""

    # Ensure enrichment columns exist
    for col in ("Exchange Code", "Ticker", "Yahoo Ticker", "Listed Country"):
        if col not in work.columns:
            work[col] = ""

    # 2) ISIN -> Yahoo symbol (parallel, with ISIN.SUFFIX filter)
    if run_isin:
        isin_mask = work["ID_Type"].eq("ISIN")
        isin_list = work.loc[isin_mask, "Stock ID"].astype(str).tolist()
        isin_to_yahoo = yahoo_symbols_from_isins_parallel(isin_list, workers=PARALLEL_WORKERS)
        work.loc[isin_mask, "Yahoo Ticker"] = work.loc[isin_mask, "Stock ID"].map(
            lambda i: clean_yahoo_symbol_from_isin(i, isin_to_yahoo.get(i))
        ).fillna("")
    else:
        isin_mask = work["ID_Type"].eq("ISIN")

    # 3) SEDOL -> Exchange Code + Ticker via OpenFIGI (runtime-only normalization)
    if run_sedol:
        sedol_mask = work["ID_Type"].eq("SEDOL")
        sedol_list = work.loc[sedol_mask, "Stock ID"].astype(str).unique().tolist()
        sedol_map = openfigi_map_sedols(sedol_list) if sedol_list else {}
        work.loc[sedol_mask, "Exchange Code"] = work.loc[sedol_mask, "Stock ID"].map(
            lambda x: (sedol_map.get(str(x)) or {}).get("exchCode", "")
        ).fillna("")
        work.loc[sedol_mask, "Ticker"] = work.loc[sedol_mask, "Stock ID"].map(
            lambda x: (sedol_map.get(str(x)) or {}).get("ticker", "")
        ).fillna("")
    else:
        sedol_mask = work["ID_Type"].eq("SEDOL")

    # 4) Country fill per ID type:
    #    - SEDOL     -> from Exchange Code (first 2 chars) via Bloomberg
    #    - BB Ticker -> from last 2 chars of Stock ID via Bloomberg
    #    - ISIN      -> from Yahoo suffix via Yahoo mapping
    ex_hits = bb_hits = yt_hits = 0
    if run_country:
        bbg_df = pd.read_csv(bbg_path, keep_default_na=False, encoding="utf-8-sig")
        yf_df  = pd.read_csv(yf_path,  keep_default_na=False, encoding="utf-8-sig")
        bbg_map = build_bbg_map(bbg_df)
        yahoo_map = build_yahoo_map(yf_df)

        # Ensure column exists, then capture BEFORE snapshot for accurate "changed this run"
        if "Listed Country" not in work.columns:
            work["Listed Country"] = ""
        before_country = work["Listed Country"].copy()

        # --- SEDOL rows: use Exchange Code (first two chars)
        sedol_rows = work["ID_Type"].eq("SEDOL")
        ex_mask = sedol_rows & work["Exchange Code"].astype(str).str.strip().ne("")
        ex_country = work.loc[ex_mask, "Exchange Code"].map(first2).map(lambda k: bbg_map.get(k, ""))
        ex_hits = int(ex_country.astype(str).str.strip().ne("").sum())
        work.loc[ex_mask & ex_country.astype(str).str.strip().ne(""), "Listed Country"] = ex_country

        # --- BB Ticker rows: use last two chars of the Stock ID (after the space)
        def bb_last2(val: str) -> str:
            s = str(val).strip()
            parts = s.split()
            if len(parts) == 2:
                return parts[-1].upper()[-2:]  # e.g., 'KS','GR','UH','C1'
            return ""
        bb_rows = work["ID_Type"].eq("BB Ticker")
        bb_mask = bb_rows & work["Stock ID"].astype(str).str.strip().ne("")
        bb_key = work.loc[bb_mask, "Stock ID"].map(bb_last2)
        bb_country = bb_key.map(lambda k: bbg_map.get(k, ""))
        bb_hits = int(bb_country.astype(str).str.strip().ne("").sum())
        work.loc[bb_mask & bb_country.astype(str).str.strip().ne(""), "Listed Country"] = bb_country

        # --- ISIN rows: use Yahoo ticker suffix mapping
        isin_rows = work["ID_Type"].eq("ISIN")
        yt_mask = isin_rows & work["Yahoo Ticker"].astype(str).str.strip().ne("")
        yt_suffix = work.loc[yt_mask, "Yahoo Ticker"].map(extract_suffix)
        yt_country = yt_suffix.map(lambda s: yahoo_map.get(s, ""))
        yt_hits = int(yt_country.astype(str).str.strip().ne("").sum())
        work.loc[yt_mask & yt_country.astype(str).str.strip().ne(""), "Listed Country"] = yt_country

        # Rows where we actually changed the country this run
        country_updates = int((work["Listed Country"] != before_country).sum())
    else:
        country_updates = 0

    # 5) Save enriched checker (optional)
    if save_enriched:
        cols_to_keep = [
            "Option Name","Asset Class Name","Name/Kind of Investment Item",
            "Stock ID","ID_Type","Listed Country","Exchange Code","Ticker","Yahoo Ticker"
        ]
        cols_exist = [c for c in cols_to_keep if c in work.columns]
        work[cols_exist].to_csv(enriched_path, index=False, na_rep="")
        print(f"📝 Enriched file saved: {enriched_path}")

    # -------------------
    # Summary (expanded, accurate)
    # -------------------
    total_nonempty = len(work)

    isin_rows  = int(work["ID_Type"].eq("ISIN").sum())
    sedol_rows = int(work["ID_Type"].eq("SEDOL").sum())
    bb_rows    = int(work["ID_Type"].eq("BB Ticker").sum())

    # Per-type enrichment “found”
    isin_found = int(work.loc[work["ID_Type"].eq("ISIN"), "Yahoo Ticker"].astype(str).str.strip().ne("").sum())
    sedol_found = int(
        work.loc[work["ID_Type"].eq("SEDOL"), ["Exchange Code","Ticker"]]
            .apply(lambda r: str(r["Exchange Code"]).strip() != "" and str(r["Ticker"]).strip() != "", axis=1)
            .sum()
    )
    bb_country_found = int(work.loc[work["ID_Type"].eq("BB Ticker"), "Listed Country"].astype(str).str.strip().ne("").sum())

    # Legacy ID enrichment metric
    id_enrichment_success = isin_found + sedol_found
    id_enrichment_rate = (id_enrichment_success / total_nonempty * 100) if total_nonempty else 0.0

    # Country mapped THIS RUN (sum of per-type hits we just computed)
    country_mapped_this_run = (ex_hits if run_country else 0) + (bb_hits if run_country else 0) + (yt_hits if run_country else 0)
    country_mapped_rate = (country_mapped_this_run / total_nonempty * 100) if total_nonempty else 0.0

    print(f"Rows with non-empty Stock ID: {total_nonempty}")
    print(f"  ISIN rows:  {isin_rows}  | Yahoo tickers found: {isin_found}")
    print(f"  SEDOL rows: {sedol_rows} | exchCode+ticker found: {sedol_found}")
    print(f"  BB rows:    {bb_rows}    | country mapped: {bb_country_found}")
    print(f"  ➤ ID enrichment success (ISIN+SEDOL): {id_enrichment_success} / {total_nonempty} = {id_enrichment_rate:.2f}%")
    print(f"  ➤ Countries resolved (this run):      {country_mapped_this_run} / {total_nonempty} = {country_mapped_rate:.2f}%")
    print(f"  ➤ Rows where 'Listed Country' CHANGED this run: {country_updates}")

    return work  # enriched rows only


def write_listed_country_back(base_path: Path, enriched_subset: pd.DataFrame, overwrite_existing: bool = True):
    """
    Write 'Listed Country' from enriched_subset back into base file, matched by exact 'Stock ID'.
    overwrite_existing=True -> replace whatever is there.
    overwrite_existing=False -> only fill blanks in base.
    """
    base = pd.read_csv(base_path, keep_default_na=False, dtype={"Stock ID": str})
    if "Stock ID" not in base.columns or "Listed Country" not in base.columns:
        raise ValueError("Base file must contain 'Stock ID' and 'Listed Country' columns.")

    if "Listed Country" not in enriched_subset.columns:
        print("No 'Listed Country' in enriched subset—nothing to write back.")
        return

    m = (
        enriched_subset[["Stock ID", "Listed Country"]]
        .dropna(subset=["Stock ID"])
        .copy()
    )
    m = m[m["Listed Country"].astype(str).str.strip().ne("")]
    m = m.drop_duplicates(subset=["Stock ID"], keep="first")

    map_country = dict(zip(m["Stock ID"].astype(str), m["Listed Country"]))

    before = base["Listed Country"].copy()
    if overwrite_existing:
        elig = base["Stock ID"].astype(str).isin(map_country.keys())
    else:
        elig = base["Stock ID"].astype(str).isin(map_country.keys()) & (
            base["Listed Country"].astype(str).str.strip().eq("")
        )

    base.loc[elig, "Listed Country"] = base.loc[elig, "Stock ID"].astype(str).map(map_country)

    changed = int((base["Listed Country"] != before).sum())
    base.to_csv(base_path, index=False, na_rep="")
    print(f"✅ Wrote 'Listed Country' back into: {base_path}")
    print(f"Rows updated in base: {changed}")


# ========= RUN (toggle as needed) =========
if __name__ == "__main__":
    enriched_subset = run_enrichment_pipeline(
        base_path=BASE_PATH,
        enriched_path=ENRICHED_PATH,
        bbg_path=BBG_PATH,
        yf_path=YF_PATH,
        run_classify=RUN_CLASSIFY,
        run_isin=RUN_ISIN_TO_YAHOO,
        run_sedol=RUN_SEDOL_TO_EXCH_TICKER,
        run_country=RUN_COUNTRY_FROM_EXCHANGE_AND_YF,
        save_enriched=SAVE_ENRICHED,
    )

    if WRITE_COUNTRY_BACK_TO_BASE:
        write_listed_country_back(
            base_path=BASE_PATH,
            enriched_subset=enriched_subset,
            overwrite_existing=OVERWRITE_EXISTING_COUNTRY,
        )


📝 Enriched file saved: D:\LinhDao\Programming\SUPERFUNdProject\final_data\hostplus_final_enriched.csv
Rows with non-empty Stock ID: 2306
  ISIN rows:  52  | Yahoo tickers found: 48
  SEDOL rows: 2252 | exchCode+ticker found: 2209
  BB rows:    0    | country mapped: 0
  ➤ ID enrichment success (ISIN+SEDOL): 2257 / 2306 = 97.88%
  ➤ Countries resolved (this run):      2257 / 2306 = 97.88%
  ➤ Rows where 'Listed Country' CHANGED this run: 0
✅ Wrote 'Listed Country' back into: D:\LinhDao\Programming\SUPERFUNdProject\final_data\hostplus_final.csv
Rows updated in base: 0
